<a href="https://colab.research.google.com/github/stvngo/Pivotal-Token-Representation-Learning/blob/main/notebooks/steering_probe_weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steering with the layer-14 probe weight vector

Compares two intervention styles on Qwen3-0.6B / GSM8K, both using the
**probe weight vector** `w` (logistic-regression coefficient at layer 14)
as the steering direction:

1. **Additive** (same form as our prior CAA loop, but with `w` instead of
   centroid-diff): `h_new = h + alpha * w`.
2. **Projection-scaling / multiplicative**: scales just the component of
   the residual stream that lies along `w`. With `v_hat = w / ||w||` and
   `proj = (h . v_hat) * v_hat`, we apply `h_new = h + (alpha - 1) * proj`.
   `alpha=0` ablates the probe direction, `alpha=1` is identity, `alpha>1`
   amplifies, `alpha=-1` flips it.

Why this matters: at layer 14 the cosine between the probe-weight vector
(norm 4.29) and the centroid-diff vector we have been using until now
(norm 9.59) is **0.155** — they are nearly orthogonal. So this experiment
probes a genuinely different axis from anything we have already tested.

We overlay our prior CAA additive curve and the random-control band for
context. Single layer (14), 100 GSM8K-test examples, paired against base.


## 1. Install dependencies

In [ ]:
%pip install -q "transformers>=4.44" datasets accelerate matplotlib seaborn scikit-learn tqdm pandas


## 2. Imports, seeding, device

In [ ]:
import json
import os
import random
import re
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

RUN_TAG = "probe_weights"
RESULTS_DIR = Path("nb_results") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE} dtype={DTYPE} run_tag={RUN_TAG}")
print(f"results -> {RESULTS_DIR.resolve()}")


## 3. Experiment configuration

In [ ]:
LAYER = 14

ADD_FACTORS = [-1.0, 0.0, 1.2, 1.4, 1.6, 1.8, 2.0]
MULT_ALPHAS = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]

MAX_EXAMPLES = 100
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.6
TOP_P = 0.9

FORCE_RERUN = False


## 4. Load the probe-weight vector and the centroid-diff vector

`w` is the logistic-regression weight learned at layer 14 (saved by
`probe_pipeline/train_sklearn.py` as `probe_weights.npy`). We also load
the centroid-diff vector `v` so we can report `cos(w, v)` — a quick
sanity check that they are not the same direction (we expect ~0.16).
Fallback chain: local repo -> GitHub raw -> manual upload.


In [ ]:
# Probe weight + centroid-diff vectors live under steering_configs/ in the
# GitHub repo (artifacts/ is otherwise gitignored).  We also accept the raw
# analysis_data/ path that probe_pipeline writes locally during training, so
# either layout works.
LOCAL_PROBE_DIR = Path("artifacts/cached3/sklearn/steering_configs")
LOCAL_ANALYSIS_DIR = Path("artifacts/cached3/sklearn/analysis_data")

PROBE_RAW_URL_BASE = (
    "https://raw.githubusercontent.com/"
    "stvngo/Pivotal-Token-Representation-Learning/"
    "main/artifacts/cached3/sklearn/steering_configs/"
)


In [ ]:
import urllib.request


def _ensure_file(local_candidates, remote_url: str, dest: Path) -> Path:
    for cand in local_candidates:
        if cand.exists():
            return cand
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(remote_url, dest)
        return dest
    except Exception as exc:
        raise RuntimeError(
            f"Could not fetch {dest.name}. Tried local: "
            f"{[str(c) for c in local_candidates]} and remote: {remote_url}. "
            f"Either clone the repo, fix PROBE_RAW_URL_BASE, or upload manually. ({exc})"
        ) from exc


def load_probe_weights(layer: int) -> np.ndarray:
    tracked_name = f"steering_layer{layer}_probe_weights.npy"
    candidates = [
        LOCAL_PROBE_DIR / tracked_name,
        LOCAL_ANALYSIS_DIR / f"layer_{layer}" / "probe_weights.npy",
    ]
    dest = RESULTS_DIR / "probes" / tracked_name
    src = _ensure_file(candidates, PROBE_RAW_URL_BASE + tracked_name, dest)
    return np.load(src).astype(np.float32).reshape(-1)


def load_centroid_diff(layer: int) -> np.ndarray:
    fname = f"steering_layer{layer}_vector.npy"
    candidates = [LOCAL_PROBE_DIR / fname]
    dest = RESULTS_DIR / "probes" / fname
    src = _ensure_file(candidates, PROBE_RAW_URL_BASE + fname, dest)
    return np.load(src).astype(np.float32).reshape(-1)


In [ ]:
# --- Google Colab: upload your own probe files ---------------------------
# Uncomment if both fallbacks above fail.
#
# from google.colab import files
# uploaded = files.upload()
# for name, data in uploaded.items():
#     dest = RESULTS_DIR / "probes" / name
#     dest.parent.mkdir(parents=True, exist_ok=True)
#     dest.write_bytes(data)


In [ ]:
w = load_probe_weights(LAYER)
v_centroid = load_centroid_diff(LAYER)

w_norm = float(np.linalg.norm(w))
v_norm = float(np.linalg.norm(v_centroid))
cos_wv = float(np.dot(w, v_centroid) / (w_norm * v_norm + 1e-12))

print(f"Probe weights w: layer={LAYER} norm={w_norm:.4f} dim={w.shape[0]}")
print(f"Centroid-diff v: layer={LAYER} norm={v_norm:.4f} dim={v_centroid.shape[0]}")
print(f"cos(w, v) = {cos_wv:+.4f}")

w_tensor = torch.from_numpy(w).to(torch.float32)
v_hat = w / max(1e-12, w_norm)
v_hat_tensor = torch.from_numpy(v_hat).to(torch.float32)


## 5. Load Qwen3-0.6B

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"

_t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(
    f"Loaded {MODEL_NAME} on {DEVICE} ({DTYPE}) "
    f"in {time.time() - _t0:.1f}s | hidden_size={model.config.hidden_size} "
    f"layers={model.config.num_hidden_layers}"
)
assert w.shape[0] == model.config.hidden_size, (
    f"probe dim {w.shape[0]} != model hidden_size {model.config.hidden_size}"
)


## 6. Steering hooks (additive and projection-scaling)

Two factories that share the same fire-on-every-forward-pass behaviour
as the existing scaffolds; the only difference is what they add to the
residual stream.


In [ ]:
import torch.nn as nn


def _get_decoder_layer(mdl: nn.Module, layer_idx: int) -> nn.Module:
    if hasattr(mdl, "model") and hasattr(mdl.model, "layers"):
        return mdl.model.layers[layer_idx]
    if hasattr(mdl, "transformer") and hasattr(mdl.transformer, "h"):
        return mdl.transformer.h[layer_idx]
    if hasattr(mdl, "decoder") and hasattr(mdl.decoder, "layers"):
        return mdl.decoder.layers[layer_idx]
    raise AttributeError(f"Cannot find decoder layers in {type(mdl)}")


def make_additive_hook(vector: torch.Tensor, alpha: float):
    """h_new = h + alpha * vector. Mirrors the existing CAA hook."""
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        v = vector.to(hidden.device).to(hidden.dtype)
        if v.dim() == 1:
            v = v.view(1, 1, -1)
        delta = alpha * v
        if isinstance(output, tuple):
            return (hidden + delta,) + output[1:]
        return hidden + delta
    return hook


def make_projection_hook(unit_vector: torch.Tensor, alpha: float):
    """h_new = h + (alpha - 1) * (h . v_hat) * v_hat. Scales only the component
    along the probe direction; other directions are untouched. alpha=1 is the
    identity, alpha=0 ablates the direction, alpha=-1 flips it."""
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        vh = unit_vector.to(hidden.device).to(hidden.dtype)
        if vh.dim() == 1:
            vh_b = vh.view(1, 1, -1)
        else:
            vh_b = vh
        proj_scalar = (hidden * vh_b).sum(dim=-1, keepdim=True)
        proj_vec = proj_scalar * vh_b
        delta = (alpha - 1.0) * proj_vec
        if isinstance(output, tuple):
            return (hidden + delta,) + output[1:]
        return hidden + delta
    return hook


def register_additive(mdl, layer: int, vector: torch.Tensor, alpha: float):
    return [_get_decoder_layer(mdl, layer).register_forward_hook(
        make_additive_hook(vector, alpha)
    )]


def register_projection(mdl, layer: int, unit_vector: torch.Tensor, alpha: float):
    return [_get_decoder_layer(mdl, layer).register_forward_hook(
        make_projection_hook(unit_vector, alpha)
    )]


def remove_hooks(handles):
    for h in handles:
        h.remove()


## 7. Answer parser + metrics

In [ ]:
def extract_gsm8k_answer(text: str):
    m = re.search(r"####\s*([+-]?\d+(?:,\d{3})*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else None


def is_correct(pred, gt):
    if pred is None or gt is None:
        return False
    p = pred.strip().replace(",", "")
    g = gt.strip().replace(",", "")
    try:
        return float(p) == float(g)
    except ValueError:
        return p == g


def compute_metrics(results):
    n = len(results)
    if n == 0:
        return {"accuracy": 0.0, "f1": 0.0, "parse_rate": 0.0, "correct": 0, "n": 0}
    correct = sum(1 for r in results if r["correct"])
    parsed = sum(1 for r in results if r["predicted"] is not None)
    tp = correct
    fp = parsed - correct
    fn = n - correct
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"accuracy": correct / n, "f1": f1, "parse_rate": parsed / n,
            "correct": correct, "n": n}


assert extract_gsm8k_answer("#### 42") == "42"
assert extract_gsm8k_answer("He has #### 1,234 apples.") == "1234"
assert extract_gsm8k_answer("...so the total is 18 apples.") == "18"
assert is_correct("3.0", "3") is True
print("Parser sanity checks passed.")


## 8. GSM8K test subset

In [ ]:
from datasets import load_dataset

_ds_full = load_dataset("openai/gsm8k", "main", split="test")
_rng = np.random.default_rng(SEED)
_indices = sorted(
    _rng.choice(len(_ds_full), size=min(MAX_EXAMPLES, len(_ds_full)), replace=False).tolist()
)
ds_subset = _ds_full.select(_indices)
examples = [
    {"question": row["question"],
     "ground_truth": extract_gsm8k_answer(row["answer"]),
     "answer_full": row["answer"]}
    for row in ds_subset
]
print(f"GSM8K test subset: {len(examples)} examples (seed={SEED})")


## 9. Generation + eval helpers

In [ ]:
def generate_once(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def _reseed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


def run_eval_with_hooks(label, register_fn, desc_prefix=""):
    _reseed()
    handles = register_fn(model) if register_fn is not None else []
    results = []
    try:
        for i, ex in enumerate(tqdm(examples, desc=f"{desc_prefix}{label}")):
            prompt = f"Question: {ex['question']}\n\nLet's think step by step.\n\n"
            text = generate_once(prompt)
            pred = extract_gsm8k_answer(text)
            ok = is_correct(pred, ex["ground_truth"])
            results.append({
                "idx": i,
                "question": ex["question"],
                "ground_truth": ex["ground_truth"],
                "predicted": pred,
                "correct": ok,
                "full_output": text[-1000:],
            })
    finally:
        remove_hooks(handles)
    return results, compute_metrics(results)


def save_run(label, results, metrics, extra_state=None):
    (RESULTS_DIR / f"{label}_results.json").write_text(json.dumps(results, indent=2))
    (RESULTS_DIR / f"{label}_generations.txt").write_text(
        "\n\n====\n\n".join(
            f"[{r['idx']}] gt={r['ground_truth']} pred={r['predicted']} correct={r['correct']}\n{r['full_output']}"
            for r in results
        )
    )
    state = {
        "label": label,
        "model": MODEL_NAME,
        "seed": SEED,
        "max_examples": len(examples),
        "max_new_tokens": MAX_NEW_TOKENS,
        "device": DEVICE,
        "run_tag": RUN_TAG,
        "metrics": metrics,
    }
    if extra_state:
        state.update(extra_state)
    (RESULTS_DIR / f"{label}_state.json").write_text(json.dumps(state, indent=2))
    return state


## 10. Base evaluation

In [ ]:
base_cache = RESULTS_DIR / "base_results.json"
if base_cache.exists() and not FORCE_RERUN:
    base_results = json.loads(base_cache.read_text())
    base_metrics = compute_metrics(base_results)
    print("Loaded cached base results.")
else:
    base_results, base_metrics = run_eval_with_hooks("base", register_fn=None)
    save_run("base", base_results, base_metrics)
print(f"base: acc={base_metrics['accuracy']:.3f} f1={base_metrics['f1']:.3f} parse={base_metrics['parse_rate']:.3f}")


## 11. Sweep: additive arm (h + alpha * w)

In [ ]:
all_runs = {"base": {"results": base_results, "metrics": base_metrics, "arm": "base"}}


def run_arm(label, register_fn, arm, factor, alpha, intervention_norm, extra=None):
    cache = RESULTS_DIR / f"{label}_results.json"
    state_path = RESULTS_DIR / f"{label}_state.json"
    if cache.exists() and state_path.exists() and not FORCE_RERUN:
        res = json.loads(cache.read_text())
        met = compute_metrics(res)
        print(f"[cached] {label}: acc={met['accuracy']:.3f} parse={met['parse_rate']:.3f}")
    else:
        res, met = run_eval_with_hooks(label, register_fn=register_fn)
        state_extra = {
            "arm": arm,
            "hook_type": arm,
            "factor": factor,
            "alpha": alpha,
            "layer": LAYER,
            "vector_type": "probe_weights",
            "vector_norm": w_norm,
            "intervention_norm": intervention_norm,
        }
        if extra:
            state_extra.update(extra)
        save_run(label, res, met, extra_state=state_extra)
        print(f"[done]   {label}: acc={met['accuracy']:.3f} parse={met['parse_rate']:.3f}")
    all_runs[label] = {
        "results": res, "metrics": met,
        "arm": arm, "factor": factor, "alpha": alpha,
        "intervention_norm": intervention_norm,
    }


for f in ADD_FACTORS:
    label = f"addw_f{f}"
    run_arm(
        label,
        register_fn=lambda mdl, a=f: register_additive(mdl, LAYER, w_tensor, a),
        arm="additive",
        factor=f,
        alpha=f,
        intervention_norm=abs(f) * w_norm,
    )


## 12. Sweep: projection-scaling arm (h + (alpha - 1) * proj_w(h))

In [ ]:
# Note: with the projection hook, the magnitude added to each token depends
# on h's existing component along v_hat, so we can't precompute a single
# intervention norm.  We log NaN and recover the actual norm from generation
# if ever needed (skipped here to keep the run cheap).
import math

for a in MULT_ALPHAS:
    label = f"projw_a{a}"
    run_arm(
        label,
        register_fn=lambda mdl, alpha=a: register_projection(mdl, LAYER, v_hat_tensor, alpha),
        arm="projection",
        factor=None,
        alpha=a,
        intervention_norm=float("nan"),
    )


## 13. Aggregate results

In [ ]:
import pandas as pd

rows = []
for name, r in all_runs.items():
    met = r["metrics"]
    rows.append({
        "run": name,
        "arm": r.get("arm", "base"),
        "factor": r.get("factor"),
        "alpha": r.get("alpha"),
        "accuracy": met["accuracy"],
        "f1": met["f1"],
        "parse_rate": met["parse_rate"],
        "intervention_norm": r.get("intervention_norm"),
    })
df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "summaries.csv", index=False)
(RESULTS_DIR / "summaries.json").write_text(json.dumps({
    n: {"accuracy": v["metrics"]["accuracy"], "f1": v["metrics"]["f1"],
        "parse_rate": v["metrics"]["parse_rate"],
        "arm": v.get("arm"), "factor": v.get("factor"), "alpha": v.get("alpha")}
    for n, v in all_runs.items()
}, indent=2))
df.sort_values(["arm", "alpha"]).reset_index(drop=True)


## 14. Plots and overlay with prior runs

We overlay:

- **CAA additive** curve from `artifacts/notebook_runs/main_gsm8k/summaries.json`
  (the existing `centroid_diff` factor sweep at layer 14).
- **Random-direction band** from `artifacts/notebook_runs/random_control/`
  (`null_summary.json`), if available.

If those local files are not present (e.g. in a fresh Colab runtime),
the overlays are silently skipped.


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

# ---- Load prior runs for overlay (optional) ------------------------------
# Tries local first, then GitHub raw (so Colab can still fetch the overlays).
PRIOR_RUNS_RAW_URL_BASE = (
    "https://raw.githubusercontent.com/"
    "stvngo/Pivotal-Token-Representation-Learning/"
    "main/"
)

def _try_load_json(rel_path):
    p = Path(rel_path)
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            return None
    try:
        with urllib.request.urlopen(PRIOR_RUNS_RAW_URL_BASE + str(rel_path)) as r:
            return json.loads(r.read().decode())
    except Exception:
        return None


caa_summary = _try_load_json("artifacts/notebook_runs/main_gsm8k/summaries.json")
random_summary = _try_load_json("artifacts/notebook_runs/random_control/null_summary.json")

caa_xy = []
if caa_summary:
    for k, m in caa_summary.items():
        if k == "base":
            continue
        try:
            f = float(k.split("_", 1)[1])
            caa_xy.append((f, m["accuracy"]))
        except Exception:
            continue
    caa_xy.sort()


# ---- Line: accuracy vs intervention parameter ----------------------------
add_rows = sorted(
    [r for r in all_runs.values() if r.get("arm") == "additive"],
    key=lambda r: r["factor"],
)
proj_rows = sorted(
    [r for r in all_runs.values() if r.get("arm") == "projection"],
    key=lambda r: r["alpha"],
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)

ax = axes[0]
ax.plot(
    [r["factor"] for r in add_rows],
    [r["metrics"]["accuracy"] for r in add_rows],
    marker="o", label="probe-weights additive (this run)",
)
if caa_xy:
    ax.plot(*zip(*caa_xy), marker="s", linestyle="--",
            label="centroid-diff additive (prior run)")
if random_summary:
    rxs = [float(k) for k in random_summary]
    rmean = [random_summary[k]["random_mean"] for k in random_summary]
    rstd = [random_summary[k]["random_std"] for k in random_summary]
    ax.errorbar(rxs, rmean, yerr=rstd, fmt="x", color="gray",
                label="random direction band (prior run)")
ax.axhline(base_metrics["accuracy"], ls=":", color="black", label="base")
ax.set_xlabel("alpha (additive)")
ax.set_ylabel("accuracy")
ax.set_title("Additive: h + alpha * w")
ax.legend(loc="best", fontsize=8)

ax = axes[1]
ax.plot(
    [r["alpha"] for r in proj_rows],
    [r["metrics"]["accuracy"] for r in proj_rows],
    marker="o", color="#2a9d8f", label="probe-weights projection-scaling",
)
ax.axhline(base_metrics["accuracy"], ls=":", color="black", label="base")
ax.axvline(1.0, ls=":", color="gray", label="alpha=1 (identity)")
ax.set_xlabel("alpha (projection scaling)")
ax.set_title("Projection-scaling: h + (alpha-1)*proj_w(h)")
ax.legend(loc="best", fontsize=8)

fig.suptitle("Probe-weight steering at layer 14 (GSM8K, n=100)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "probe_weights_curves.png")
plt.show()


## 15. Bar chart: matched intervention strength

Three matched comparisons:

- **amplify**: additive `alpha=1.2` vs projection `alpha=2.0` (both
  amplify the probe direction roughly 2x relative to the natural
  component already present at the layer).
- **ablate**: additive `alpha=0` vs projection `alpha=0` (the additive
  case is a no-op since the perturbation is `0`; the projection case
  removes the component along `w`).
- **flip**: additive `alpha=-1` vs projection `alpha=-1`.


In [ ]:
def acc_for(arm, key, val):
    """Return accuracy for the run matching (arm, key=val), or NaN if missing.

    Returning NaN (instead of None) lets matplotlib silently skip missing
    bars rather than crashing with `int + NoneType` in `_convert_units`.
    """
    for r in all_runs.values():
        if r.get("arm") != arm:
            continue
        v = r.get(key)
        if v is None:
            continue
        if abs(float(v) - val) < 1e-9:
            return float(r["metrics"]["accuracy"])
    return float("nan")


bars = []
for tag, add_alpha, proj_alpha in (
    ("amplify", 1.2, 2.0),
    ("ablate",  0.0, 0.0),
    ("flip",   -1.0, -1.0),
):
    bars.append({
        "tag": tag,
        "additive_alpha": add_alpha,
        "projection_alpha": proj_alpha,
        "additive": acc_for("additive", "factor", add_alpha),
        "projection": acc_for("projection", "alpha", proj_alpha),
        "base": base_metrics["accuracy"],
    })

bars_df = pd.DataFrame(bars)

missing = []
for b in bars:
    if not np.isfinite(b["additive"]):
        missing.append(f"additive @ factor={b['additive_alpha']}")
    if not np.isfinite(b["projection"]):
        missing.append(f"projection @ alpha={b['projection_alpha']}")
if missing:
    print("[warn] missing matched runs (bars will be blank):")
    for m in missing:
        print(f"  - {m}")

add_y = np.array([b["additive"] for b in bars], dtype=float)
proj_y = np.array([b["projection"] for b in bars], dtype=float)

fig, ax = plt.subplots(figsize=(7.5, 4))
xs = np.arange(len(bars))
w_bar = 0.35
ax.bar(xs - w_bar/2, add_y, w_bar,
       label="additive (probe-weights)", color="#4c72b0")
ax.bar(xs + w_bar/2, proj_y, w_bar,
       label="projection-scaling (probe-weights)", color="#2a9d8f")
ax.axhline(base_metrics["accuracy"], ls="--", color="gray", label="base")
ax.set_xticks(xs)
ax.set_xticklabels([b["tag"] for b in bars])
ax.set_ylabel("accuracy")
ax.set_title("Additive vs projection-scaling at matched alpha")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "matched_intervention_bars.png")
plt.show()
bars_df


## 16. Energy / norm table

In [ ]:
# For the additive arm: ||delta|| = |alpha| * ||w||  (constant per token).
# For the projection-scaling arm: ||delta|| varies per token; we report the
# theoretical max in the worst case ||delta|| <= |alpha-1| * ||h_par|| <=
# |alpha-1| * ||h||, but more useful is to record the design knob.
energy_rows = []
for r in all_runs.values():
    if r.get("arm") == "additive":
        energy_rows.append({
            "arm": "additive",
            "param": r["factor"],
            "param_name": "alpha (factor)",
            "intervention_norm_per_token": abs(r["factor"]) * w_norm,
            "accuracy": r["metrics"]["accuracy"],
            "lift_pp": (r["metrics"]["accuracy"] - base_metrics["accuracy"]) * 100,
        })
    elif r.get("arm") == "projection":
        energy_rows.append({
            "arm": "projection",
            "param": r["alpha"],
            "param_name": "alpha",
            "intervention_norm_per_token": float("nan"),
            "accuracy": r["metrics"]["accuracy"],
            "lift_pp": (r["metrics"]["accuracy"] - base_metrics["accuracy"]) * 100,
        })

energy_df = pd.DataFrame(energy_rows).sort_values(["arm", "param"]).reset_index(drop=True)
energy_df.to_csv(RESULTS_DIR / "energy_table.csv", index=False)
energy_df


## 17. Flip overlap with base

In [ ]:
base_idx_correct = {r["idx"]: r["correct"] for r in base_results}


def flips_for(label):
    res = all_runs[label]["results"]
    return {r["idx"] for r in res if base_idx_correct.get(r["idx"]) != r["correct"]}


def gained_for(label):
    res = all_runs[label]["results"]
    return {r["idx"] for r in res
            if not base_idx_correct.get(r["idx"], False) and r["correct"]}


def lost_for(label):
    res = all_runs[label]["results"]
    return {r["idx"] for r in res
            if base_idx_correct.get(r["idx"], False) and not r["correct"]}


flip_summary = {}
for label, r in all_runs.items():
    if label == "base":
        continue
    flip_summary[label] = {
        "gained": sorted(gained_for(label)),
        "lost": sorted(lost_for(label)),
        "n_gained": len(gained_for(label)),
        "n_lost": len(lost_for(label)),
        "n_flips": len(flips_for(label)),
    }
(RESULTS_DIR / "flip_summary.json").write_text(json.dumps(flip_summary, indent=2))


# Compare additive(alpha=1.2) vs projection(alpha=2.0) flips: are they hitting the same examples?
add_label = "addw_f1.2"
proj_label = "projw_a2.0"
if add_label in all_runs and proj_label in all_runs:
    a_flips = flips_for(add_label)
    p_flips = flips_for(proj_label)
    union = a_flips | p_flips
    inter = a_flips & p_flips
    jacc = len(inter) / max(1, len(union))
    print(f"flip overlap, {add_label} vs {proj_label}: jaccard={jacc:.3f}, "
          f"|add_only|={len(a_flips - p_flips)} |proj_only|={len(p_flips - a_flips)}")
    (RESULTS_DIR / "flip_overlap_add_vs_proj.json").write_text(json.dumps({
        "add_label": add_label,
        "proj_label": proj_label,
        "jaccard": jacc,
        "intersection": sorted(inter),
        "additive_only": sorted(a_flips - p_flips),
        "projection_only": sorted(p_flips - a_flips),
    }, indent=2))
print("Wrote flip_summary.json + flip_overlap_add_vs_proj.json")


## 18. Bundle and (optionally) download

In [ ]:
zip_path = Path(f"nb_results_{RUN_TAG}.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(RESULTS_DIR.parent))
print(f"Zipped {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Not on Colab — the zip is on the local filesystem.")
